# Schema Matcher LLM V6 — Test New Trained Model

This notebook evaluates your **newly trained** fine-tuned **Phi-3-mini + LoRA** model on real-world schema mapping scenarios.

**Configure below:** Set `OUTPUT_DIR` to your training output folder (e.g. `schema_matcher_llm`). The notebook will auto-detect `adapter/` or the latest `checkpoint-*/`.

**V6 Enhancements (on top of V5):**
- **Two-level transform output**: `transform_type` (15 high-level) + `sub_operation` (30+ fine-grained)
- Sub-operations include: `add`, `subtract`, `multiply`, `divide`, `extract_year`, `extract_month`, `age_calculation`, `duration_days`, `status_flag`, `code_to_label`, `fk_dimension_lookup`, `multi_hop_lookup`, `string_template`, etc.
- Auto-inference of sub_operation from mapping context
- Enhanced reasoning templates with sub_operation signals

**V5 Enhancements:**
- Explicit transform decision rules (rename vs fk_lookup, code_to_label vs conditional)
- Value column vs join key column selection rules
- Contrastive training examples, cross-table column selection training
- LoRA rank 96 with all 7 target modules

**What it tests:**
- **Candidate Selection** — Does the model pick the correct source column(s)?
- **Transformation Declaration** — Does it identify the right transform type?
- **Sub-Operation** — Does it output the correct fine-grained operation?
- **Cross-table joins, ambiguity, noise columns, and complex transforms**

**4 Domains x 22 Test Cases:**
| Domain | Tests | Transforms Covered |
|--------|-------|--------------------| 
| HR / Employee | 5 | rename, concat, date_part, fk_lookup |
| E-commerce | 6 | fk_lookup, concat, date_part, conditional, arithmetic, code_to_label |
| Healthcare | 6 | concat, date_part, code_to_label, fk_lookup, date_diff, lookup_join |
| Banking (noisy) | 5 | concat, code_to_label, conditional, bucketing, date_part |

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q torch transformers peft accelerate

## Cell 2 — Configuration (New Model)

Set `OUTPUT_DIR` to your training output folder (e.g. `schema_matcher_llm`). The notebook will auto-detect:
- `adapter/` (merged LoRA adapter), or
- latest `checkpoint-*/` if no adapter folder exists.

Alternatively, set `ADAPTER_PATH` directly to override auto-detection.

In [ ]:
import os, re, time, json, glob
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ──────────────────────────────────────────────
#  NEW MODEL: Set your training output directory
# ──────────────────────────────────────────────
OUTPUT_DIR = "schema_matcher_llm"   # folder containing adapter/ or checkpoint-*/
# Or set ADAPTER_PATH directly to override auto-detection:
ADAPTER_PATH = None   # leave None to auto-detect from OUTPUT_DIR
BASE_MODEL   = "microsoft/Phi-3-mini-4k-instruct"

# Auto-resolve adapter path
if ADAPTER_PATH is None:
    adapter_path = os.path.join(OUTPUT_DIR, "adapter")
    checkpoint_glob = os.path.join(OUTPUT_DIR, "checkpoint-*")
    if os.path.isdir(adapter_path):
        ADAPTER_PATH = adapter_path
        print(f"Using adapter: {ADAPTER_PATH}")
    else:
        def _ckpt_num(p):
            m = re.search(r"checkpoint-(\d+)", p)
            return int(m.group(1)) if m else 0
        checkpoints = sorted(glob.glob(checkpoint_glob), key=_ckpt_num)
        if checkpoints:
            ADAPTER_PATH = checkpoints[-1]
            print(f"Using latest checkpoint: {ADAPTER_PATH}")
        else:
            raise FileNotFoundError(f"No adapter/ or checkpoint-* found in {OUTPUT_DIR}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Cell 3 — Load Model + LoRA Adapter

In [ ]:
print("[1/3] Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[2/3] Loading base model: {BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=False,
    attn_implementation="eager",
)

print(f"[3/3] Loading LoRA adapter: {ADAPTER_PATH}")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded on {DEVICE}  |  {total_params:,} params")

## Cell 4 — System Prompt & Helper Functions

In [ ]:
SYSTEM_PROMPT = """You are a schema mapping expert. Given a source database schema and a target column, identify which source column(s) should be mapped to the target and what transformation is needed.

COLUMN SELECTION RULES:
- Use ONLY columns that exist in the source schema
- Choose the MINIMUM number of source columns needed
- ALWAYS output the VALUE column(s) that hold the actual data, NOT the FK/join key columns used to traverse between tables. Example: to get a department name via a join, output departments.dept_name, NOT the employees.department_id FK column
- When a mapping requires columns from MULTIPLE tables (e.g. date_diff between dates in different tables), select the actual data columns from each table, not the FK columns that link them
- For surrogate keys, use the primary key of the matching entity table

TRANSFORM DECISION RULES:
- rename: ONLY when the source column is in the SAME table as the target's primary entity and no computation is needed. If a join/FK is required to reach the column, it is NOT a rename
- direct_copy: identical to rename but emphasizes no name change
- fk_lookup: when the source column is in a DIFFERENT table reached via a single FK join. The key distinction from rename is that a join is required
- lookup_join: when the source column requires traversing 2+ joins (multi-hop). Use this instead of fk_lookup for multi-table chains
- concat: combining 2+ columns into one string value
- date_part: extracting year, month, day, etc. from a date/datetime
- date_diff: computing the difference between two dates (duration, age)
- date_format: converting a date to a display string format
- date_parse: parsing a string into a date/datetime type
- arithmetic: math operations (add, subtract, multiply, divide) between columns. Do NOT assume arithmetic from column names alone -- if salary maps to annual_salary with no formula specified, use rename
- conditional: deriving a BOOLEAN true/false flag from a status/code column. The target must be boolean/flag type
- code_to_label: mapping a code/status column to a human-readable LABEL string (e.g. gender_code -> gender_label, status_code -> status_name). This is NOT the same as conditional -- code_to_label produces a label string, conditional produces a boolean
- bucketing: grouping a numeric/continuous value into range categories
- type_cast: converting data type without changing the value
- template: assembling columns into a formatted display string pattern

Valid transform types: rename, direct_copy, concat, fk_lookup, date_part, date_diff, date_format, date_parse, arithmetic, conditional, bucketing, code_to_label, type_cast, lookup_join, template

SUB-OPERATION: For each transform, specify the fine-grained operation:
- rename: rename_only
- direct_copy: direct_copy
- concat: concat_two, concat_multi (3+ columns)
- fk_lookup: fk_dimension_lookup
- lookup_join: multi_hop_lookup
- date_part: extract_year, extract_month, extract_day, extract_quarter, extract_hour
- date_diff: date_difference, age_calculation, duration_days, duration_hours
- date_format: format_date
- date_parse: parse_date
- arithmetic: add, subtract, multiply, divide, ratio_percentage, scaling_unit_conversion
- conditional: threshold_flag, status_flag, equality_check, null_presence_flag
- code_to_label: code_to_label, category_harmonization
- bucketing: bucketing_binning, range_classification
- type_cast: type_cast_numeric, type_cast_string, type_cast_date
- template: string_template

OUTPUT RULES:
- Respond with EXACTLY ONE mapping for the requested target column
- Do NOT output mappings for other columns

Respond in EXACTLY this format:
source_columns: <table.column>, <table.column>, ...
transform_type: <transform>
sub_operation: <fine-grained operation>
reasoning: <brief explanation>"""


@torch.no_grad()
def generate(prompt, max_new_tokens=300):
    """Send a mapping query to the model and return raw text."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        text = (
            f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
            f"<|user|>\n{prompt}<|end|>\n"
            f"<|assistant|>\n"
        )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def parse(response):
    """Extract source_columns, transform_type, sub_operation, reasoning from model output."""
    result = {"source_columns": [], "transform_type": "", "sub_operation": "", "reasoning": ""}
    m = re.search(r'source_columns?:\s*(.+?)(?:\n|$)', response, re.IGNORECASE)
    if m:
        result["source_columns"] = [
            c.strip() for c in m.group(1).split(",") if c.strip()
        ]
    m = re.search(r'transform_type:\s*(\S+)', response, re.IGNORECASE)
    if m:
        result["transform_type"] = m.group(1).strip().lower()
    m = re.search(r'sub_operation:\s*(\S+)', response, re.IGNORECASE)
    if m:
        result["sub_operation"] = m.group(1).strip().lower()
    m = re.search(r'reasoning:\s*(.+)', response, re.IGNORECASE | re.DOTALL)
    if m:
        result["reasoning"] = m.group(1).strip()[:200]
    return result

print("Helper functions ready.")

## Cell 5 — Define Test Scenarios (4 Domains, 22 Cases)

In [ ]:
TESTS = [
    # ═══════════════════════════════════════════════
    #  SCENARIO 1 — HR / Employee Management
    # ═══════════════════════════════════════════════
    {
        "name": "HR Schema",
        "schema": (
            "Source Schema:\n"
            "  employees(emp_id PK int, first_name string, last_name string, "
            "email string, hire_date date, salary decimal, department_id FK int)\n"
            "  departments(dept_id PK int, dept_name string, location string, budget decimal)\n"
            "\n"
            "Joins:\n"
            "  employees.department_id = departments.dept_id"
        ),
        "targets": [
            ('dim_employee.employee_key (int) - "surrogate key"',
             ["employees.emp_id"], "rename"),
            ('dim_employee.full_name (string) - "employee full name"',
             ["employees.first_name", "employees.last_name"], "concat"),
            ('dim_employee.hire_year (int) - "year hired"',
             ["employees.hire_date"], "date_part"),
            ('dim_employee.department_name (string) - "department name"',
             ["departments.dept_name"], "fk_lookup"),
            ('dim_employee.annual_salary (decimal) - "yearly salary"',
             ["employees.salary"], "rename"),
        ],
    },

    # ═══════════════════════════════════════════════
    #  SCENARIO 2 — E-commerce / Orders
    # ═══════════════════════════════════════════════
    {
        "name": "E-commerce Schema",
        "schema": (
            "Source Schema:\n"
            "  orders(order_id PK int, customer_id FK int, order_date date, "
            "total_amount decimal, status string)\n"
            "  customers(customer_id PK int, first_name string, last_name string, "
            "email string, country string)\n"
            "  order_items(item_id PK int, order_id FK int, product_id FK int, "
            "quantity int, unit_price decimal)\n"
            "  products(product_id PK int, product_name string, category_code string, "
            "weight_grams int)\n"
            "\n"
            "Joins:\n"
            "  orders.customer_id = customers.customer_id\n"
            "  order_items.order_id = orders.order_id\n"
            "  order_items.product_id = products.product_id"
        ),
        "targets": [
            ('fact_order.customer_country (string) - "country of the customer"',
             ["customers.country"], "fk_lookup"),
            ('fact_order.customer_name (string) - "full name of customer"',
             ["customers.first_name", "customers.last_name"], "concat"),
            ('fact_order.order_year (int) - "year the order was placed"',
             ["orders.order_date"], "date_part"),
            ('fact_order.is_completed (boolean) - "whether order is complete"',
             ["orders.status"], "conditional"),
            ('fact_sales.line_total (decimal) - "quantity times unit price"',
             ["order_items.quantity", "order_items.unit_price"], "arithmetic"),
            ('dim_product.category_label (string) - "human readable category"',
             ["products.category_code"], "code_to_label"),
        ],
    },

    # ═══════════════════════════════════════════════
    #  SCENARIO 3 — Healthcare
    # ═══════════════════════════════════════════════
    {
        "name": "Healthcare Schema",
        "schema": (
            "Source Schema:\n"
            "  patients(patient_id PK int, first_name string, last_name string, "
            "dob date, gender_code string, insurance_id FK int)\n"
            "  insurance_plans(plan_id PK int, plan_name string, provider_name string, "
            "coverage_type_code string)\n"
            "  visits(visit_id PK int, patient_id FK int, doctor_id FK int, "
            "visit_date date, diagnosis_code string, visit_cost decimal)\n"
            "  doctors(doctor_id PK int, first_name string, last_name string, "
            "specialization_code string, department_id FK int)\n"
            "  hospital_departments(dept_id PK int, dept_name string, floor int)\n"
            "\n"
            "Joins:\n"
            "  patients.insurance_id = insurance_plans.plan_id\n"
            "  visits.patient_id = patients.patient_id\n"
            "  visits.doctor_id = doctors.doctor_id\n"
            "  doctors.department_id = hospital_departments.dept_id"
        ),
        "targets": [
            ('dim_patient.patient_name (string) - "full name of patient"',
             ["patients.first_name", "patients.last_name"], "concat"),
            ('dim_patient.birth_year (int) - "year patient was born"',
             ["patients.dob"], "date_part"),
            ('dim_patient.gender_label (string) - "gender in readable form"',
             ["patients.gender_code"], "code_to_label"),
            ('dim_patient.insurance_plan (string) - "name of patient insurance plan"',
             ["insurance_plans.plan_name"], "fk_lookup"),
            ('fact_visit.patient_age_at_visit (int) - "patient age at time of visit"',
             ["patients.dob", "visits.visit_date"], "date_diff"),
            ('fact_visit.department_name (string) - "department from doctor lookup"',
             ["hospital_departments.dept_name"], "lookup_join"),
        ],
    },

    # ═══════════════════════════════════════════════
    #  SCENARIO 4 — Banking (with noisy audit columns)
    # ═══════════════════════════════════════════════
    {
        "name": "Banking Schema (with noise columns)",
        "schema": (
            "Source Schema:\n"
            "  bank_accounts(account_id PK int, account_number string, "
            "account_type_code string, customer_id FK int, open_date date, "
            "balance decimal, status_code string, created_by string, updated_at datetime)\n"
            "  bank_customers(customer_id PK int, first_name string, last_name string, "
            "email string, credit_score int, income_annual decimal, "
            "risk_category_code string, created_by string, updated_at datetime)\n"
            "  bank_transactions(txn_id PK int, account_id FK int, txn_date date, "
            "amount decimal, txn_type_code string, description string, created_by string)\n"
            "\n"
            "Joins:\n"
            "  bank_accounts.customer_id = bank_customers.customer_id\n"
            "  bank_transactions.account_id = bank_accounts.account_id"
        ),
        "targets": [
            ('dim_customer.customer_name (string) - "customer full name"',
             ["bank_customers.first_name", "bank_customers.last_name"], "concat"),
            ('dim_account.account_type_label (string) - "human readable account type"',
             ["bank_accounts.account_type_code"], "code_to_label"),
            ('dim_account.is_active (boolean) - "whether account is currently open"',
             ["bank_accounts.status_code"], "conditional"),
            ('dim_customer.income_bracket (string) - "income range bucket"',
             ["bank_customers.income_annual"], "bucketing"),
            ('fact_txn.transaction_year (int) - "year of transaction"',
             ["bank_transactions.txn_date"], "date_part"),
        ],
    },
]

total_tests = sum(len(t["targets"]) for t in TESTS)
print(f"Loaded {total_tests} test cases across {len(TESTS)} schemas.")

## Cell 6 — Run All Tests

In [ ]:
# ── Accumulators ──
total = 0
col_correct = 0
tf_correct  = 0
both_correct = 0
sub_op_total = 0
sub_op_present = 0
total_time  = 0.0
all_results = []   # detailed log for every test
failures    = []

print("=" * 80)
print(f"  CANDIDATE SELECTION + TRANSFORMATION TEST  ({total_tests} cases)")
print("=" * 80)

for scenario in TESTS:
    print(f"\n{'─'*60}")
    print(f"  {scenario['name']}")
    print(f"{'─'*60}")

    for target_str, exp_cols, exp_tf in scenario["targets"]:
        total += 1
        prompt = f"{scenario['schema']}\n\nMap target: {target_str}"

        t0 = time.time()
        response = generate(prompt)
        elapsed = time.time() - t0
        total_time += elapsed

        parsed    = parse(response)
        pred_cols = sorted([c.lower().strip() for c in parsed["source_columns"]])
        expected  = sorted([c.lower().strip() for c in exp_cols])
        pred_tf   = parsed["transform_type"]
        pred_sub  = parsed.get("sub_operation", "")

        cols_ok = pred_cols == expected
        tf_ok   = pred_tf == exp_tf

        if cols_ok:  col_correct  += 1
        if tf_ok:    tf_correct   += 1
        if cols_ok and tf_ok: both_correct += 1

        sub_op_total += 1
        if pred_sub:
            sub_op_present += 1

        status = "PASS" if (cols_ok and tf_ok) else "FAIL"
        target_short = target_str.split(" - ")[0] if " - " in target_str else target_str[:50]

        # ── Per-case output ──
        print(f"  [{status}] {target_short}")
        print(f"         Cols     : {pred_cols}  {'✓' if cols_ok else '✗  expected ' + str(expected)}")
        print(f"         Transform: {pred_tf}  {'✓' if tf_ok else '✗  expected ' + exp_tf}")
        if pred_sub:
            print(f"         Sub-Op   : {pred_sub}")
        if parsed.get("reasoning"):
            print(f"         Reasoning: {parsed['reasoning'][:120]}")
        print(f"         Time     : {elapsed:.2f}s")

        # ── Save detailed record ──
        record = {
            "scenario":       scenario["name"],
            "target":         target_short,
            "expected_cols":  expected,
            "predicted_cols": pred_cols,
            "cols_correct":   cols_ok,
            "expected_tf":    exp_tf,
            "predicted_tf":   pred_tf,
            "predicted_sub_op": pred_sub,
            "tf_correct":     tf_ok,
            "both_correct":   cols_ok and tf_ok,
            "reasoning":      parsed.get("reasoning", ""),
            "raw_response":   response,
            "time_seconds":   round(elapsed, 3),
        }
        all_results.append(record)

        if not (cols_ok and tf_ok):
            failures.append(record)

print("\n" + "=" * 80)
print("  Tests complete.")
print("=" * 80)

## Cell 7 — Results Summary

In [ ]:
n = total

print("=" * 80)
print("  RESULTS SUMMARY")
print("=" * 80)
print(f"  Source column accuracy : {col_correct}/{n}  =  {col_correct/n:.1%}")
print(f"  Transform accuracy    : {tf_correct}/{n}  =  {tf_correct/n:.1%}")
print(f"  Both correct          : {both_correct}/{n}  =  {both_correct/n:.1%}")
print(f"  Sub-op present        : {sub_op_present}/{sub_op_total}  =  {sub_op_present/max(sub_op_total,1):.1%}")
print(f"  Average response time : {total_time/n:.2f}s")
print(f"  Total time            : {total_time:.1f}s")
print("=" * 80)

# ── Sub-operation distribution ──
from collections import Counter
sub_op_dist = Counter(r["predicted_sub_op"] for r in all_results if r.get("predicted_sub_op"))
if sub_op_dist:
    print("\n  Sub-Operation Distribution:")
    print(f"  {'Sub-Operation':<30} {'Count':>6}")
    print(f"  {'─'*30} {'─'*6}")
    for so, cnt in sub_op_dist.most_common():
        print(f"  {so:<30} {cnt:>6}")

# ── Per-scenario breakdown ──
print("\n  Per-Scenario Breakdown:")
print(f"  {'Scenario':<40} {'Cols':>6} {'Trans':>6} {'Both':>6}")
print(f"  {'─'*40} {'─'*6} {'─'*6} {'─'*6}")
for scenario in TESTS:
    sc_results = [r for r in all_results if r["scenario"] == scenario["name"]]
    sc_n = len(sc_results)
    sc_cols  = sum(1 for r in sc_results if r["cols_correct"])
    sc_tf    = sum(1 for r in sc_results if r["tf_correct"])
    sc_both  = sum(1 for r in sc_results if r["both_correct"])
    print(f"  {scenario['name']:<40} {sc_cols}/{sc_n:>3}  {sc_tf}/{sc_n:>3}  {sc_both}/{sc_n:>3}")

# ── Per-transform breakdown ──
print("\n  Per-Transform Breakdown:")
tf_types = sorted(set(r["expected_tf"] for r in all_results))
print(f"  {'Transform':<20} {'Total':>6} {'Cols OK':>8} {'TF OK':>8} {'Both':>8}")
print(f"  {'─'*20} {'─'*6} {'─'*8} {'─'*8} {'─'*8}")
for tf in tf_types:
    tf_res = [r for r in all_results if r["expected_tf"] == tf]
    tf_n = len(tf_res)
    c_ok = sum(1 for r in tf_res if r["cols_correct"])
    t_ok = sum(1 for r in tf_res if r["tf_correct"])
    b_ok = sum(1 for r in tf_res if r["both_correct"])
    print(f"  {tf:<20} {tf_n:>6} {c_ok:>8} {t_ok:>8} {b_ok:>8}")

## Cell 8 — Failure Analysis

In [ ]:
if failures:
    print(f"\n{'='*80}")
    print(f"  FAILURES — {len(failures)} / {n}")
    print(f"{'='*80}")
    for i, f in enumerate(failures, 1):
        print(f"\n  [{i}] {f['scenario']}  →  {f['target']}")
        if not f["cols_correct"]:
            print(f"      Cols : predicted {f['predicted_cols']}")
            print(f"             expected  {f['expected_cols']}")
        if not f["tf_correct"]:
            print(f"      Trans: predicted '{f['predicted_tf']}'  expected '{f['expected_tf']}'")
        print(f"      Reasoning: {f['reasoning'][:150]}")
        print(f"      Raw response:")
        for line in f["raw_response"].split("\n")[:6]:
            print(f"        {line}")
else:
    print("\n  🎉  ALL 22 TESTS PASSED — No failures!")

## Cell 9 — Export Results to JSON

In [ ]:
export = {
    "summary": {
        "total_tests":        n,
        "col_accuracy":       round(col_correct / n, 4),
        "transform_accuracy": round(tf_correct / n, 4),
        "both_accuracy":      round(both_correct / n, 4),
        "avg_time_seconds":   round(total_time / n, 3),
        "total_time_seconds": round(total_time, 1),
        "device":             DEVICE,
        "gpu":                torch.cuda.get_device_name(0) if DEVICE == "cuda" else "N/A",
        "adapter_path":       ADAPTER_PATH,
        "base_model":         BASE_MODEL,
    },
    "results": all_results,
}

out_path = "candidate_selection_test_results.json"
with open(out_path, "w") as fp:
    json.dump(export, fp, indent=2)

print(f"Results exported to: {out_path}")
print(json.dumps(export["summary"], indent=2))

## Cell 10 — Quick Sanity: Single Interactive Query

Use this cell to test any custom mapping interactively.

In [ ]:
# ── Try your own mapping query ──
custom_prompt = """Source Schema:
  students(student_id PK int, first_name string, last_name string, enrollment_date date, gpa decimal, major_code string, advisor_id FK int)
  professors(prof_id PK int, first_name string, last_name string, department string)

Joins:
  students.advisor_id = professors.prof_id

Map target: dim_student.advisor_name (string) - "full name of the student's advisor"""

t0 = time.time()
resp = generate(custom_prompt)
elapsed = time.time() - t0

print(f"Response ({elapsed:.2f}s):\n")
print(resp)
print(f"\nParsed: {parse(resp)}")